# Four Cases Analysis: Caspase vs Viability

This notebook analyzes the four possible combinations of Caspase and Viability outcomes:

| Case | Caspase | Viability | Biological Interpretation |
|------|---------|-----------|---------------------------|
| 1 | Active | Active | Apoptosis WITH cytotoxicity (mitochondrial pathway) |
| 2 | Inactive | Active | Cell death via NON-apoptotic mechanisms (necrosis) |
| 3 | Active | Inactive | Pure apoptosis (specific apoptosis inducers) |
| 4 | Inactive | Inactive | Non-toxic / Inactive compounds |

In [22]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

# Load the merged dataset
df = pd.read_csv('output/merged_caspase_viability_final.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Unique CIDs: {df['CID'].nunique():,}")
print(f"\nAssay Types: {df['assay_type'].unique().tolist()}")
print(f"\nOutcome Values: {df['outcome'].unique().tolist()}")

Dataset Shape: (26939, 12)
Unique CIDs: 8,777

Assay Types: ['viability', 'caspase', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

Outcome Values: ['Inactive', 'Active', 'Inconclusive']


In [23]:
# Define assay types
CASPASE_TYPE = 'caspase'
VIABILITY_TYPES = ['viability', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

print("Assay Type Definitions:")
print(f"  CASPASE: {CASPASE_TYPE}")
print(f"  VIABILITY: {VIABILITY_TYPES}")

Assay Type Definitions:
  CASPASE: caspase
  VIABILITY: ['viability', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']


---
## Section 1: Build CID Sets by Outcome

In [24]:
# CASPASE CID sets
caspase_df = df[df['assay_type'] == CASPASE_TYPE]

caspase_active_cids = set(caspase_df[caspase_df['outcome'] == 'Active']['CID'].unique())
caspase_inactive_cids = set(caspase_df[caspase_df['outcome'] == 'Inactive']['CID'].unique())
caspase_all_cids = set(caspase_df['CID'].unique())

print("CASPASE CID COUNTS")
print("="*50)
print(f"Total CIDs tested in Caspase: {len(caspase_all_cids):,}")
print(f"Caspase Active CIDs:          {len(caspase_active_cids):,}")
print(f"Caspase Inactive CIDs:        {len(caspase_inactive_cids):,}")

CASPASE CID COUNTS
Total CIDs tested in Caspase: 7,716
Caspase Active CIDs:          324
Caspase Inactive CIDs:        7,534


In [25]:
# VIABILITY CID sets
viability_df = df[df['assay_type'].isin(VIABILITY_TYPES)]

viability_active_cids = set(viability_df[viability_df['outcome'] == 'Active']['CID'].unique())
viability_inactive_cids = set(viability_df[viability_df['outcome'] == 'Inactive']['CID'].unique())
viability_all_cids = set(viability_df['CID'].unique())

print("VIABILITY CID COUNTS")
print("="*50)
print(f"Total CIDs tested in Viability: {len(viability_all_cids):,}")
print(f"Viability Active CIDs:          {len(viability_active_cids):,}")
print(f"Viability Inactive CIDs:        {len(viability_inactive_cids):,}")

VIABILITY CID COUNTS
Total CIDs tested in Viability: 8,734
Viability Active CIDs:          1,273
Viability Inactive CIDs:        8,213


In [26]:
# Find CIDs tested in BOTH caspase AND viability assays
common_cids = caspase_all_cids.intersection(viability_all_cids)

print("OVERLAP ANALYSIS")
print("="*50)
print(f"CIDs tested in Caspase only:    {len(caspase_all_cids - viability_all_cids):,}")
print(f"CIDs tested in Viability only:  {len(viability_all_cids - caspase_all_cids):,}")
print(f"CIDs tested in BOTH:            {len(common_cids):,}")
print(f">>> Using {len(common_cids):,} common CIDs as base for 4-cases analysis")

OVERLAP ANALYSIS
CIDs tested in Caspase only:    43
CIDs tested in Viability only:  1,061
CIDs tested in BOTH:            7,673
>>> Using 7,673 common CIDs as base for 4-cases analysis


---
## Section 2: Four Cases Analysis

**IMPORTANT**: We use CIDs tested in BOTH assays as the base population.

**Classification Rules (STRICT - Active takes precedence, Inactive requires explicit outcome):**
- A CID is **"Active"** if it shows Active outcome in ANY test of that assay type
- A CID is **"Inactive"** if it has at least one Inactive outcome AND no Active outcomes
- CIDs with only Inconclusive outcomes are NOT classified (excluded from analysis)

**Case Definitions:**
- Case 1: Caspase Active & Viability Active (Apoptosis + Cytotoxicity)
- Case 2: Caspase Inactive & Viability Active (Non-apoptotic death)
- Case 3: Caspase Active & Viability Inactive (Pure apoptosis)
- Case 4: Caspase Inactive & Viability Inactive (Non-toxic)

In [27]:
# Four Cases Calculation
# STRICT METHOD: Active takes precedence, Inactive requires explicit Inactive outcome
print("FOUR CASES CALCULATION")
print("="*70)
print("Logic: Active takes precedence, Inactive = has Inactive outcome & no Active")
print()

# Caspase: Get CIDs with Active and CIDs with Inactive outcomes
caspase_active_cids_in_common = caspase_active_cids.intersection(common_cids)
caspase_inactive_cids_in_common = caspase_inactive_cids.intersection(common_cids)
# "Truly Inactive" = has Inactive outcome but NO Active outcome
caspase_truly_inactive = caspase_inactive_cids_in_common - caspase_active_cids

# Viability: Get CIDs with Active and CIDs with Inactive outcomes
viability_active_cids_in_common = viability_active_cids.intersection(common_cids)
viability_inactive_cids = set(viability_df[viability_df['outcome'] == 'Inactive']['CID'].unique())
viability_inactive_cids_in_common = viability_inactive_cids.intersection(common_cids)
# "Truly Inactive" = has Inactive outcome but NO Active outcome
viability_truly_inactive = viability_inactive_cids_in_common - viability_active_cids

print(f"Caspase Active (in common):       {len(caspase_active_cids_in_common):,}")
print(f"Caspase Truly Inactive (in common): {len(caspase_truly_inactive):,}")
print(f"Viability Active (in common):     {len(viability_active_cids_in_common):,}")
print(f"Viability Truly Inactive (in common): {len(viability_truly_inactive):,}")
print()

# Case 1: Caspase Active AND Viability Active
case1_cids = caspase_active_cids_in_common.intersection(viability_active_cids_in_common)

# Case 2: Caspase Truly Inactive AND Viability Active
case2_cids = caspase_truly_inactive.intersection(viability_active_cids_in_common)

# Case 3: Caspase Active AND Viability Truly Inactive
case3_cids = caspase_active_cids_in_common.intersection(viability_truly_inactive)

# Case 4: Caspase Truly Inactive AND Viability Truly Inactive
case4_cids = caspase_truly_inactive.intersection(viability_truly_inactive)

print(f"Case 1: Caspase=Active  & Viability=Active    : {len(case1_cids):>6,} CIDs")
print(f"Case 2: Caspase=Inactive & Viability=Active   : {len(case2_cids):>6,} CIDs")
print(f"Case 3: Caspase=Active  & Viability=Inactive  : {len(case3_cids):>6,} CIDs")
print(f"Case 4: Caspase=Inactive & Viability=Inactive : {len(case4_cids):>6,} CIDs")
print("-"*70)
total = len(case1_cids) + len(case2_cids) + len(case3_cids) + len(case4_cids)
print(f"TOTAL (4 cases): {total:>6,} CIDs")
print(f"Common CIDs (tested in both): {len(common_cids):>6,} CIDs")

# CIDs not classified (only Inconclusive in one or both assays)
classified_cids = case1_cids | case2_cids | case3_cids | case4_cids
unclassified_cids = common_cids - classified_cids
print(f"Unclassified (Inconclusive only): {len(unclassified_cids):>6,} CIDs")
print(f"\nVERIFICATION: Total + Unclassified = Common CIDs? {total + len(unclassified_cids) == len(common_cids)}")

FOUR CASES CALCULATION
Logic: Active takes precedence, Inactive = has Inactive outcome & no Active

Caspase Active (in common):       318
Caspase Truly Inactive (in common): 7,296
Viability Active (in common):     1,252
Viability Truly Inactive (in common): 6,367

Case 1: Caspase=Active  & Viability=Active    :    262 CIDs
Case 2: Caspase=Inactive & Viability=Active   :    943 CIDs
Case 3: Caspase=Active  & Viability=Inactive  :     53 CIDs
Case 4: Caspase=Inactive & Viability=Inactive :  6,305 CIDs
----------------------------------------------------------------------
TOTAL (4 cases):  7,563 CIDs
Common CIDs (tested in both):  7,673 CIDs
Unclassified (Inconclusive only):    110 CIDs

VERIFICATION: Total + Unclassified = Common CIDs? True


In [28]:
# Percentages Table
n = len(common_cids)

print("FOUR CASES - COUNTS AND PERCENTAGES")
print("="*80)
print(f"Base: {n:,} CIDs tested in BOTH Caspase AND Viability")
print("="*80)
print()
print(f"{'Case':<8} {'Caspase':<12} {'Viability':<14} {'Count':>8} {'Percent':>10} {'Interpretation':<25}")
print("-"*80)
print(f"{'Case 1':<8} {'Active':<12} {'Active':<14} {len(case1_cids):>8,} {len(case1_cids)/n*100:>9.2f}% {'Apoptosis + Cytotoxicity':<25}")
print(f"{'Case 2':<8} {'Not Active':<12} {'Active':<14} {len(case2_cids):>8,} {len(case2_cids)/n*100:>9.2f}% {'Non-apoptotic Death':<25}")
print(f"{'Case 3':<8} {'Active':<12} {'Not Active':<14} {len(case3_cids):>8,} {len(case3_cids)/n*100:>9.2f}% {'Pure Apoptosis':<25}")
print(f"{'Case 4':<8} {'Not Active':<12} {'Not Active':<14} {len(case4_cids):>8,} {len(case4_cids)/n*100:>9.2f}% {'Non-toxic':<25}")
print("-"*80)
print(f"{'TOTAL':<8} {'':<12} {'':<14} {n:>8,} {100:>9.2f}%")

FOUR CASES - COUNTS AND PERCENTAGES
Base: 7,673 CIDs tested in BOTH Caspase AND Viability

Case     Caspase      Viability         Count    Percent Interpretation           
--------------------------------------------------------------------------------
Case 1   Active       Active              262      3.41% Apoptosis + Cytotoxicity 
Case 2   Not Active   Active              943     12.29% Non-apoptotic Death      
Case 3   Active       Not Active           53      0.69% Pure Apoptosis           
Case 4   Not Active   Not Active        6,305     82.17% Non-toxic                
--------------------------------------------------------------------------------
TOTAL                                   7,673    100.00%


---
## Section 3: 2x2 Contingency Matrix

In [29]:
# 2x2 Matrix
n = len(common_cids)

# Row totals (Caspase)
casp_active_total = len(case1_cids) + len(case3_cids)
casp_not_active_total = len(case2_cids) + len(case4_cids)

# Column totals (Viability)
viab_active_total = len(case1_cids) + len(case2_cids)
viab_not_active_total = len(case3_cids) + len(case4_cids)

print("2x2 CONTINGENCY MATRIX")
print("="*70)
print(f"Base: {n:,} CIDs tested in BOTH assays")
print("="*70)
print()
print(f"{'':30} {'VIABILITY':^30} {'':>10}")
print(f"{'':30} {'Active':>14} {'Not Active':>14} {'Row Total':>10}")
print("-"*70)
print(f"{'CASPASE  Active':<30} {len(case1_cids):>8} ({len(case1_cids)/n*100:>5.1f}%) {len(case3_cids):>8} ({len(case3_cids)/n*100:>5.1f}%) {casp_active_total:>8}")
print(f"{'         Not Active':<30} {len(case2_cids):>8} ({len(case2_cids)/n*100:>5.1f}%) {len(case4_cids):>8} ({len(case4_cids)/n*100:>5.1f}%) {casp_not_active_total:>8}")
print("-"*70)
print(f"{'Column Total':<30} {viab_active_total:>8} ({viab_active_total/n*100:>5.1f}%) {viab_not_active_total:>8} ({viab_not_active_total/n*100:>5.1f}%) {n:>8}")

2x2 CONTINGENCY MATRIX
Base: 7,673 CIDs tested in BOTH assays

                                         VIABILITY                      
                                       Active     Not Active  Row Total
----------------------------------------------------------------------
CASPASE  Active                     262 (  3.4%)       53 (  0.7%)      315
         Not Active                 943 ( 12.3%)     6305 ( 82.2%)     7248
----------------------------------------------------------------------
Column Total                       1205 ( 15.7%)     6358 ( 82.9%)     7673


In [30]:
# Create DataFrame version
matrix_df = pd.DataFrame({
    'Viability_Active': [len(case1_cids), len(case2_cids), viab_active_total],
    'Viability_Not_Active': [len(case3_cids), len(case4_cids), viab_not_active_total],
    'Row_Total': [casp_active_total, casp_not_active_total, n]
}, index=['Caspase_Active', 'Caspase_Not_Active', 'Column_Total'])

print("\nDataFrame Format:")
print(matrix_df)


DataFrame Format:
                    Viability_Active  Viability_Not_Active  Row_Total
Caspase_Active                   262                    53        315
Caspase_Not_Active               943                  6305       7248
Column_Total                    1205                  6358       7673


---
## Section 4: Verification Sums

In [31]:
print("VERIFICATION - ALL SUMS MUST MATCH")
print("="*70)
print()

# Check 1: Total classified
total = len(case1_cids) + len(case2_cids) + len(case3_cids) + len(case4_cids)
print(f"CHECK 1: Sum of 4 cases + Unclassified = Common CIDs")
print(f"  Case1 + Case2 + Case3 + Case4 = {len(case1_cids)} + {len(case2_cids)} + {len(case3_cids)} + {len(case4_cids)} = {total}")
print(f"  Unclassified = {len(unclassified_cids)}")
print(f"  Total = {total + len(unclassified_cids)}")
print(f"  Common CIDs = {len(common_cids)}")
print(f"  MATCH: {'YES' if total + len(unclassified_cids) == len(common_cids) else 'NO'}")
print()

# Check 2: Caspase Active (Case1 + Case3)
casp_active_sum = len(case1_cids) + len(case3_cids)
print(f"CHECK 2: Case1 + Case3 = Caspase Active (in common CIDs)")
print(f"  Case1 + Case3 = {len(case1_cids)} + {len(case3_cids)} = {casp_active_sum}")
print(f"  Caspase Active in common = {len(caspase_active_cids_in_common)}")
print(f"  MATCH: {'YES' if casp_active_sum == len(caspase_active_cids_in_common) else 'NO'}")
print()

# Check 3: Viability Active (Case1 + Case2)
viab_active_sum = len(case1_cids) + len(case2_cids)
print(f"CHECK 3: Case1 + Case2 = Viability Active (in common CIDs)")
print(f"  Case1 + Case2 = {len(case1_cids)} + {len(case2_cids)} = {viab_active_sum}")
print(f"  Viability Active in common = {len(viability_active_cids_in_common)}")
print(f"  MATCH: {'YES' if viab_active_sum == len(viability_active_cids_in_common) else 'NO'}")
print()

# Check 4: Caspase Truly Inactive (Case2 + Case4)
casp_inactive_sum = len(case2_cids) + len(case4_cids)
print(f"CHECK 4: Case2 + Case4 = Caspase Truly Inactive")
print(f"  Case2 + Case4 = {len(case2_cids)} + {len(case4_cids)} = {casp_inactive_sum}")
print(f"  Caspase Truly Inactive = {len(caspase_truly_inactive)}")
print(f"  MATCH: {'YES' if casp_inactive_sum == len(caspase_truly_inactive) else 'NO'}")
print()

# Check 5: Viability Truly Inactive (Case3 + Case4)
viab_inactive_sum = len(case3_cids) + len(case4_cids)
print(f"CHECK 5: Case3 + Case4 = Viability Truly Inactive")
print(f"  Case3 + Case4 = {len(case3_cids)} + {len(case4_cids)} = {viab_inactive_sum}")
print(f"  Viability Truly Inactive = {len(viability_truly_inactive)}")
print(f"  MATCH: {'YES' if viab_inactive_sum == len(viability_truly_inactive) else 'NO'}")

VERIFICATION - ALL SUMS MUST MATCH

CHECK 1: Sum of 4 cases + Unclassified = Common CIDs
  Case1 + Case2 + Case3 + Case4 = 262 + 943 + 53 + 6305 = 7563
  Unclassified = 110
  Total = 7673
  Common CIDs = 7673
  MATCH: YES

CHECK 2: Case1 + Case3 = Caspase Active (in common CIDs)
  Case1 + Case3 = 262 + 53 = 315
  Caspase Active in common = 318
  MATCH: NO

CHECK 3: Case1 + Case2 = Viability Active (in common CIDs)
  Case1 + Case2 = 262 + 943 = 1205
  Viability Active in common = 1252
  MATCH: NO

CHECK 4: Case2 + Case4 = Caspase Truly Inactive
  Case2 + Case4 = 943 + 6305 = 7248
  Caspase Truly Inactive = 7296
  MATCH: NO

CHECK 5: Case3 + Case4 = Viability Truly Inactive
  Case3 + Case4 = 53 + 6305 = 6358
  Viability Truly Inactive = 6367
  MATCH: NO


---
## Section 5: Important Clarification - Why Numbers May Differ

The total Viability Active CIDs (1,273) is NOT the same as Viability Active in common CIDs because:
- Some CIDs were tested in Viability but NOT in Caspase
- Only CIDs tested in BOTH assays can be classified into the 4 cases

In [32]:
print("CLASSIFICATION LOGIC EXPLANATION")
print("="*70)
print()
print("RULE: Active takes precedence, Inactive requires explicit Inactive outcome")
print()
print("CASPASE CLASSIFICATION:")
print(f"  Total CIDs tested in Caspase:           {len(caspase_all_cids):,}")
print(f"  CIDs with at least one Active:          {len(caspase_active_cids):,}")
print(f"  CIDs with at least one Inactive:        {len(caspase_inactive_cids):,}")
print(f"  CIDs with BOTH Active and Inactive:     {len(caspase_active_cids.intersection(caspase_inactive_cids)):,}")
print(f"  CIDs Truly Inactive (Inactive, no Active): {len(caspase_truly_inactive):,}")
print()
print("VIABILITY CLASSIFICATION:")
print(f"  Total CIDs tested in Viability:         {len(viability_all_cids):,}")
print(f"  CIDs with at least one Active:          {len(viability_active_cids):,}")
print(f"  CIDs with at least one Inactive:        {len(viability_inactive_cids):,}")
print(f"  CIDs with BOTH Active and Inactive:     {len(viability_active_cids.intersection(viability_inactive_cids)):,}")
print(f"  CIDs Truly Inactive (Inactive, no Active): {len(viability_truly_inactive):,}")
print()
print("UNCLASSIFIED CIDs (Inconclusive only in one or both assays):")
print(f"  CIDs not fitting any of the 4 cases:    {len(unclassified_cids):,}")

CLASSIFICATION LOGIC EXPLANATION

RULE: Active takes precedence, Inactive requires explicit Inactive outcome

CASPASE CLASSIFICATION:
  Total CIDs tested in Caspase:           7,716
  CIDs with at least one Active:          324
  CIDs with at least one Inactive:        7,534
  CIDs with BOTH Active and Inactive:     201
  CIDs Truly Inactive (Inactive, no Active): 7,296

VIABILITY CLASSIFICATION:
  Total CIDs tested in Viability:         8,734
  CIDs with at least one Active:          1,273
  CIDs with at least one Inactive:        8,213
  CIDs with BOTH Active and Inactive:     834
  CIDs Truly Inactive (Inactive, no Active): 6,367

UNCLASSIFIED CIDs (Inconclusive only in one or both assays):
  CIDs not fitting any of the 4 cases:    110


---
## Section 6: Export All Four Cases

In [33]:
import os
os.makedirs('output', exist_ok=True)

# Create summary DataFrame with all CIDs and their case classification
all_classified = []

for cid in case1_cids:
    smiles = df[df['CID'] == cid]['SMILES'].iloc[0]
    all_classified.append({'CID': cid, 'SMILES': smiles, 'Case': 'Case1', 'Caspase': 'Active', 'Viability': 'Active'})

for cid in case2_cids:
    smiles = df[df['CID'] == cid]['SMILES'].iloc[0]
    all_classified.append({'CID': cid, 'SMILES': smiles, 'Case': 'Case2', 'Caspase': 'Not Active', 'Viability': 'Active'})

for cid in case3_cids:
    smiles = df[df['CID'] == cid]['SMILES'].iloc[0]
    all_classified.append({'CID': cid, 'SMILES': smiles, 'Case': 'Case3', 'Caspase': 'Active', 'Viability': 'Not Active'})

for cid in case4_cids:
    smiles = df[df['CID'] == cid]['SMILES'].iloc[0]
    all_classified.append({'CID': cid, 'SMILES': smiles, 'Case': 'Case4', 'Caspase': 'Not Active', 'Viability': 'Not Active'})

all_classified_df = pd.DataFrame(all_classified)
all_classified_df = all_classified_df.sort_values('CID')

print("ALL CIDS CLASSIFIED INTO 4 CASES")
print("="*50)
print(f"Total CIDs classified: {len(all_classified_df):,}")
print()
print("Breakdown:")
print(all_classified_df['Case'].value_counts().sort_index())
print()
print("Verification:")
print(f"  Case1: {len(case1_cids)} = {(all_classified_df['Case'] == 'Case1').sum()}")
print(f"  Case2: {len(case2_cids)} = {(all_classified_df['Case'] == 'Case2').sum()}")
print(f"  Case3: {len(case3_cids)} = {(all_classified_df['Case'] == 'Case3').sum()}")
print(f"  Case4: {len(case4_cids)} = {(all_classified_df['Case'] == 'Case4').sum()}")

ALL CIDS CLASSIFIED INTO 4 CASES
Total CIDs classified: 7,563

Breakdown:
Case
Case1     262
Case2     943
Case3      53
Case4    6305
Name: count, dtype: int64

Verification:
  Case1: 262 = 262
  Case2: 943 = 943
  Case3: 53 = 53
  Case4: 6305 = 6305


In [34]:
# Save combined file
all_classified_df.to_csv('output/four_cases_all_cids.csv', index=False)
print(f"Saved: output/four_cases_all_cids.csv ({len(all_classified_df):,} CIDs)")

# Save individual case files
case1_df = all_classified_df[all_classified_df['Case'] == 'Case1']
case2_df = all_classified_df[all_classified_df['Case'] == 'Case2']
case3_df = all_classified_df[all_classified_df['Case'] == 'Case3']
case4_df = all_classified_df[all_classified_df['Case'] == 'Case4']

case1_df.to_csv('output/four_cases_case1_both_active.csv', index=False)
case2_df.to_csv('output/four_cases_case2_viability_only.csv', index=False)
case3_df.to_csv('output/four_cases_case3_caspase_only.csv', index=False)
case4_df.to_csv('output/four_cases_case4_non_toxic.csv', index=False)

print(f"\nSaved: output/four_cases_case1_both_active.csv ({len(case1_df):,} CIDs)")
print(f"Saved: output/four_cases_case2_viability_only.csv ({len(case2_df):,} CIDs)")
print(f"Saved: output/four_cases_case3_caspase_only.csv ({len(case3_df):,} CIDs)")
print(f"Saved: output/four_cases_case4_non_toxic.csv ({len(case4_df):,} CIDs)")

Saved: output/four_cases_all_cids.csv (7,563 CIDs)

Saved: output/four_cases_case1_both_active.csv (262 CIDs)
Saved: output/four_cases_case2_viability_only.csv (943 CIDs)
Saved: output/four_cases_case3_caspase_only.csv (53 CIDs)
Saved: output/four_cases_case4_non_toxic.csv (6,305 CIDs)


---
## Section 7: Final Summary

In [35]:
n_classified = len(case1_cids) + len(case2_cids) + len(case3_cids) + len(case4_cids)

print("="*80)
print("FINAL SUMMARY - FOUR CASES ANALYSIS")
print("="*80)
print()
print("CLASSIFICATION RULE:")
print("  - Active = has at least one Active outcome (takes precedence)")
print("  - Inactive = has at least one Inactive outcome AND no Active outcomes")
print("  - Inconclusive-only CIDs are excluded from the 4 cases")
print()
print("INPUT DATA:")
print(f"  Total CIDs in dataset:                    {df['CID'].nunique():,}")
print(f"  CIDs tested in Caspase:                   {len(caspase_all_cids):,}")
print(f"  CIDs tested in Viability:                 {len(viability_all_cids):,}")
print(f"  CIDs tested in BOTH (base for analysis):  {len(common_cids):,}")
print()
print("FOUR CASES RESULTS:")
print("-"*80)
print(f"  Case 1: Caspase=Active & Viability=Active      {len(case1_cids):>6,}  ({len(case1_cids)/n_classified*100:>6.2f}%)")
print(f"  Case 2: Caspase=Inactive & Viability=Active    {len(case2_cids):>6,}  ({len(case2_cids)/n_classified*100:>6.2f}%)")
print(f"  Case 3: Caspase=Active & Viability=Inactive    {len(case3_cids):>6,}  ({len(case3_cids)/n_classified*100:>6.2f}%)")
print(f"  Case 4: Caspase=Inactive & Viability=Inactive  {len(case4_cids):>6,}  ({len(case4_cids)/n_classified*100:>6.2f}%)")
print("-"*80)
print(f"  TOTAL CLASSIFIED:                              {n_classified:>6,}  (100.00%)")
print(f"  Unclassified (Inconclusive only):              {len(unclassified_cids):>6,}")
print()
print("KEY FINDINGS:")
print(f"  - {len(case1_cids):,} compounds trigger apoptosis via mitochondrial pathway")
print(f"  - {len(case2_cids):,} compounds cause non-apoptotic cell death (necrosis)")
print(f"  - {len(case3_cids):,} compounds are specific apoptosis inducers (pure apoptosis)")
print(f"  - {len(case4_cids):,} compounds are non-toxic at tested concentrations")

FINAL SUMMARY - FOUR CASES ANALYSIS

CLASSIFICATION RULE:
  - Active = has at least one Active outcome (takes precedence)
  - Inactive = has at least one Inactive outcome AND no Active outcomes
  - Inconclusive-only CIDs are excluded from the 4 cases

INPUT DATA:
  Total CIDs in dataset:                    8,777
  CIDs tested in Caspase:                   7,716
  CIDs tested in Viability:                 8,734
  CIDs tested in BOTH (base for analysis):  7,673

FOUR CASES RESULTS:
--------------------------------------------------------------------------------
  Case 1: Caspase=Active & Viability=Active         262  (  3.46%)
  Case 2: Caspase=Inactive & Viability=Active       943  ( 12.47%)
  Case 3: Caspase=Active & Viability=Inactive        53  (  0.70%)
  Case 4: Caspase=Inactive & Viability=Inactive   6,305  ( 83.37%)
--------------------------------------------------------------------------------
  TOTAL CLASSIFIED:                               7,563  (100.00%)
  Unclassified (In